In [ ]:
from mssm.models import *
from mssmViz.sim import *
from mssmViz.plot import *
from mssmViz.extract import eval_coverage
from mssm.src.python.compare import compare_CDL
import pickle
import copy
import os
import dotenv
dotenv.load_dotenv()

n_cores = int(os.getenv("n_cores"))

from defaults import (
    default_gamm_kwargs,
    default_gammlss_kwargs,
    default_gsmm_kwargs,
    default_comparison_kwargs,
)

default_gamm_kwargs["n_cores"] = n_cores
default_gammlss_kwargs["n_cores"] = n_cores
default_gsmm_kwargs["n_cores"] = n_cores

# To speed things up
default_gamm_kwargs["method"] = "Chol"
default_gammlss_kwargs["method"] = "Chol"
default_gsmm_kwargs["method"] = "Chol"

qefs_kwargs = copy.deepcopy(default_gsmm_kwargs)
qefs_kwargs["method"] = "qEFS"
qefs_kwargs["max_inner"] = 2000

uncor_kwargs = copy.deepcopy(default_comparison_kwargs)
uncor_kwargs["correct_V"] = False
uncor_kwargs["correct_t1"] = True

n_sim = 100
n_ranef = 40
n_c = 10 # Number of effect strength increases

In [ ]:
############################# Simulation 4: General #############################
sim_fams = [MultiGauss(3,[Identity() for _ in range(3)]),
            MULNOMLSS(4),
            ScaledT(),
            PropHaz([0],[0])]
fam_names = ["MGauss", "Multinomial", "ScaledT", "PropHaz"]

for should_correlate in [False]:

    for fam_name, sim_fam in zip(fam_names,sim_fams):
        
        # Set up storage for current sim
        Failures = np.zeros((1 + 2, n_c))
        AIC_rej = np.zeros((3 + 2, n_c))

        # Assign correct family for qEFS estimator
        if isinstance(sim_fam,ScaledT):
            gsmm_fam = GAMLSSGSMMFamily(1,sim_fam)
        elif isinstance(sim_fam,MULNOMLSS):
            gsmm_fam = GAMLSSGSMMFamily(4,sim_fam)
        else:
            gsmm_fam = sim_fam

        for c_i,c_val in enumerate(np.linspace(0,1,n_c)):
        
            iterator = tqdm(range(n_sim),desc="Simulating",leave=True)
            for sim_i in iterator:

                # Get data for different families
                if isinstance(sim_fam,ScaledT):
                    sim_dat = sim4(500,2,c=c_val,seed=sim_i,family=sim_fam,
                            binom_offset = 0,n_ranef=n_ranef,
                            correlate=should_correlate)
                    
                    sim_dat.to_csv((f"./results/data/sim5_gen/sim_size:{n_sim}_fam:"
                                    f"{fam_name}_corr:{should_correlate}_set:{sim_i}_c:{c_i}.csv"),index=False)
                
                    sim_dat = pd.read_csv((f"./results/data/sim5_gen/sim_size:{n_sim}_fam:"
                                           f"{fam_name}_corr:{should_correlate}_set:{sim_i}_c:{c_i}.csv"))
                    
                    # We need to model only the mean: \mu_i
                    sim_formula_m = Formula(lhs("y"),
                                        [i(),f(["x0"]),f(["x1"]),f(["x2"]),f(["x3"]),ri("x4")],
                                        data=sim_dat)
                    
                    # Second formula excludes ri("x4")
                    sim_formula_m2 = Formula(lhs("y"),
                                        [i(),f(["x0"]),f(["x1"]),f(["x2"]),f(["x3"])],
                                        data=sim_dat)
                    
                    sim_formulas = [sim_formula_m]
                    sim_formulas2 = [sim_formula_m2]

                    fcoef = [0,*np.sort(sim_formulas[0].coef_idx_per_term[5]).tolist()] # a+ ri("x4")
                    fcoef2 = [0] # a

                elif isinstance(sim_fam,PropHaz):
                    qefs_kwargs["n_cores"] = 1 # Bug in multiprocess fork on MacOs
                    sim_dat = sim4(500,2,c=c_val,seed=sim_i,family=sim_fam,
                            binom_offset = 0.1,n_ranef=n_ranef,
                            correlate=should_correlate)
                    sim_dat = sim_dat.sort_values(['y'],ascending=[False])
                    sim_dat = sim_dat.reset_index(drop=True)
                    
                    sim_dat.to_csv((f"./results/data/sim5_gen/sim_size:{n_sim}_fam:"
                                    f"{fam_name}_corr:{should_correlate}_set:{sim_i}_c:{c_i}.csv"),index=False)
                
                    sim_dat = pd.read_csv((f"./results/data/sim5_gen/sim_size:{n_sim}_fam:"
                                           f"{fam_name}_corr:{should_correlate}_set:{sim_i}_c:{c_i}.csv"))

                    u,inv = np.unique(sim_dat["y"],return_inverse=True)
                    ut = np.flip(u)
                    r = np.abs(inv - max(inv))
                    sim_fam = PropHaz(ut=ut,r=r)
                    gsmm_fam = copy.deepcopy(sim_fam)

                    # Cannot have intercept!
                    sim_formula_m = Formula(lhs("delta"),
                                        [f(["x0"]),f(["x1"]),f(["x2"]),f(["x3"]),ri("x4")],
                                        data=sim_dat)
                    
                    sim_formula_m2 = Formula(lhs("delta"),
                                        [f(["x0"]),f(["x1"]),f(["x2"]),f(["x3"])],
                                        data=sim_dat)
                    
                    sim_formulas = [sim_formula_m]
                    sim_formulas2 = [sim_formula_m2]

                    fcoef = [0,*np.sort(sim_formulas[0].coef_idx_per_term[4]).tolist()] #ri("x4")
                    fcoef2 = [0]
                
                elif isinstance(sim_fam,MULNOMLSS):
                    # We need to specify K-1 formulas - see the `MULNOMLSS` docstring for details.
                    sim_dat = sim22(500, 2, c=c_val, seed=sim_i,
                                    correlate=should_correlate,n_ranef=n_ranef)
                    
                    sim_dat.to_csv((f"./results/data/sim5_gen/sim_size:{n_sim}_fam:"
                                    f"{fam_name}_corr:{should_correlate}_set:{sim_i}_c:{c_i}.csv"),index=False)
                
                    sim_dat = pd.read_csv((f"./results/data/sim5_gen/sim_size:{n_sim}_fam:"
                                           f"{fam_name}_corr:{should_correlate}_set:{sim_i}_c:{c_i}.csv"))

                    sim_formulas = []
                    for k in range(4):
                        terms = [i(),f([f"x{k}"]),ri("x4")] if k == 0 else [i(),f([f"x{k}"])]
                        sim_formulas.append(Formula(lhs("y"),terms, data=sim_dat))

                    sim_formulas2 = []
                    for k in range(4):
                        sim_formulas2.append(Formula(lhs("y"),[i(),f([f"x{k}"])], data=sim_dat))
                    
                    fcoef = [0,*np.sort(sim_formulas[0].coef_idx_per_term[2]).tolist()] #a + ri("x4")
                    fcoef2 = [0] #a
                else:
                    # Multi Gauss case
                    sim_dat = sim23(500, c=c_val, seed=sim_i,
                                    correlate=should_correlate, n_ranef=n_ranef)
                    
                    sim_dat.to_csv((f"./results/data/sim5_gen/sim_size:{n_sim}_fam:"
                                    f"{fam_name}_corr:{should_correlate}_set:{sim_i}_c:{c_i}.csv"),index=False)
                
                    sim_dat = pd.read_csv((f"./results/data/sim5_gen/sim_size:{n_sim}_fam:"
                                           f"{fam_name}_corr:{should_correlate}_set:{sim_i}_c:{c_i}.csv"))

                    # We need formulas for each mean
                    sim_formulas = [
                        Formula(lhs("y0"), [i(), f(["x0"]), ri("x4")], data=sim_dat),
                        Formula(lhs("y1"), [i(), f(["x1"]), f(["x2"])], data=sim_dat),
                        Formula(lhs("y2"), [i(), f(["x3"])], data=sim_dat),
                    ]

                    sim_formulas2 = [
                        Formula(lhs("y0"), [i(), f(["x0"])], data=sim_dat),
                        Formula(lhs("y1"), [i(), f(["x1"]), f(["x2"])], data=sim_dat),
                        Formula(lhs("y2"), [i(), f(["x3"])], data=sim_dat),
                    ]

                    fcoef = [0,*np.sort(sim_formulas[0].coef_idx_per_term[2]).tolist()] #a + ri("x4")
                    fcoef2 = [0] #a
                
                
                sim_i_failed = [False, *[False for _ in range(2)]]

                ############################# Fit model with EFS #############################
                if isinstance(sim_fam,ScaledT):
                    model_efs = GAMM(sim_formulas[0],copy.deepcopy(sim_fam))
                    model_efs2 = GAMM(sim_formulas2[0],copy.deepcopy(sim_fam))
                    model_kwargs = copy.deepcopy(default_gamm_kwargs)
                elif isinstance(sim_fam,MULNOMLSS):
                    model_efs = GAMMLSS(sim_formulas,sim_fam)
                    model_efs2 = GAMMLSS(sim_formulas2,sim_fam)
                    model_kwargs = copy.deepcopy(default_gammlss_kwargs)
                else:
                    model_efs = GSMM(sim_formulas,sim_fam)
                    model_efs2 = GSMM(sim_formulas2,sim_fam)
                    model_kwargs = copy.deepcopy(default_gsmm_kwargs)

                try:
                    with warnings.catch_warnings():
                        warnings.simplefilter("ignore")
                        model_efs.fit(**model_kwargs)
                        model_efs2.fit(**model_kwargs)
                except:
                    sim_i_failed[0] = True

                ############################# Fit model with qEFS #############################
                models = [model_efs]
                models2 = [model_efs2]
                for rai in range(2):
                    gsmm_model = GSMM(formulas=sim_formulas,family=gsmm_fam)
                    gsmm_model2 = GSMM(formulas=sim_formulas2,family=gsmm_fam)
                    
                    try:
                        with warnings.catch_warnings():
                            warnings.simplefilter("ignore")
                            qefs_kwargs["structured_qefs_budget"] = fcoef if rai > 0 else 0
                            gsmm_model.fit(**qefs_kwargs)
                            qefs_kwargs["structured_qefs_budget"] = fcoef2 if rai > 0 else 0
                            gsmm_model2.fit(**qefs_kwargs)
                    except:
                        sim_i_failed[1+rai] = True

                    models.append(gsmm_model)
                    models2.append(gsmm_model2)
                
                ######################################## Compare ####################################
                comp_idx = 0
                for mi, (model1, model2) in enumerate(zip(models,models2)):

                    if sim_i_failed[mi]:
                        print(f"Models {mi+1} failed at {sim_i}")
                        Failures[mi,c_i] += 1
                        continue

                    # Not converged but not failed outright
                    if model1.info.code > 0 or model2.info.code > 0:
                        Failures[mi,c_i] += 1
                    
                    #plot(model)
                    if mi == 0:
                        # Uncorrected aic
                        uncor_result = compare_CDL(model1,model2,**uncor_kwargs)

                        if uncor_result["aic_diff"] < 0:
                            AIC_rej[comp_idx,c_i] += 1
                        comp_idx += 1

                        # Compute heuristic upper edf from WPS (2016) as well
                        llk1 = uncor_result["aic1"] - 2*uncor_result["DOF12"]
                        llk2 = uncor_result["aic2"] - 2*uncor_result["DOF22"] 
                        aic1_t1 =  llk1 + 2*uncor_result["DOF1"]
                        aic2_t1 =  llk2 + 2*uncor_result["DOF2"]

                        aic_diff_t1 = aic1_t1 - aic2_t1

                        if aic_diff_t1 < 0:
                            AIC_rej[comp_idx,c_i] += 1
                        comp_idx += 1

                    # Corrected aic
                    cor_result = compare_CDL(model1,model2,**default_comparison_kwargs)

                    if cor_result["aic_diff"] < 0:
                        AIC_rej[comp_idx,c_i] += 1
                    comp_idx += 1

                ###################################### Save in progress results ######################################
                res = {"aic":AIC_rej,
                       "Failures":Failures,
                    }
                
                # Show some progress info
                progress = [np.round(AIC_rej[0,c_i]/(sim_i + 1),decimals=2),
                            np.round(AIC_rej[1,c_i]/(sim_i + 1),decimals=2),
                            np.round(AIC_rej[2,c_i]/(sim_i + 1),decimals=2),
                            *[np.round(AIC_rej[comp_idx,c_i]/(sim_i + 1),decimals=2)
                              for comp_idx in range(3,AIC_rej.shape[0])]]
                
                iterator.set_description_str(desc=f"Simulating c={np.round(c_val,decimals=2)} Acc.: {progress}", refresh=True)
                
                with open(f'./results/sim/sim5_gen/size:{n_sim}_fam:{fam_name}_corr:{should_correlate}.pickle', 'wb') as file:
                    pickle.dump(res,file, protocol=pickle.HIGHEST_PROTOCOL)
            
            iterator.close()